# Metacognitive Medical Digital Twins Pipeline

This notebook implements the complete MDT pipeline with MIMIC-IV and Medical-O1 data sources.

**Alignment Components:**
- Theory of Mind module for user belief inference
- Composite reward engine (5 components: safety, empathy, proactivity, metacognition, semantic)
- GRPO training for multi-objective alignment

**Ontology Components:**
- LOINC, SNOMED-CT, ICD-10 code mappings
- Clinical reference ranges validation
- MIMIC-IV item ID mappings

**Data Sources:**
- MIMIC-IV (ICU trajectories)
- Medical-O1 (reasoning chains)


## 1. Setup & Environment

In [1]:
# Environment setup
import sys
sys.path.append(".")

# Core imports
from config.configs import DataConfig
from data.mimic_processor import MIMICProcessor
from data.medical_o1_processor import MedicalO1Processor
from core.theory_of_mind import TheoryOfMindModule
from rewards.composite_engine import CompositeRewardEngine
from training.grpo_trainer import run_grpo_training
from utils.ontology_validator import OntologyValidator
from utils.helpers import clean_memory, setup_logging

print("✓ All components imported successfully")


✓ All components imported successfully


## 2. Configuration

In [ ]:
# Setup logging
setup_logging()
import logging
logger = logging.getLogger(__name__)

logger.info("="*80)
logger.info("MEDICAL DIGITAL TWIN - MASTER PIPELINE")
logger.info("="*80)

# Load configuration
data_config = DataConfig()
print(f"✓ Configuration loaded")
print(f"  MIMIC patients: {data_config.max_patients}")
print(f"  Medical-O1 examples: {data_config.max_o1_examples}")


: 

## 3. Data Loading

In [ ]:
# Load MIMIC-IV data
print("Loading MIMIC-IV data...")
mimic_processor = MIMICProcessor(data_config)

if mimic_processor.check_availability():
    mimic_data = mimic_processor.process_all_patients(
        max_patients=data_config.max_patients
    )
    print(f"✓ Loaded {len(mimic_data)} MIMIC examples")
else:
    print("⚠️ MIMIC data not available")
    mimic_data = []

: 

## 4. Model Training (SFT)

In [ ]:
# Import training components
from training.sft_trainer import run_sft_training
from models.mdt_model import MedicalDigitalTwinModel

# Initialize model
model = MedicalDigitalTwinModel()

# Run SFT training
print("Starting SFT training...")
trained_model = run_sft_training(
    model=model,
    train_data=all_training_data,
    config=data_config
)

print("✓ SFT training completed")


: 

## 5. Alignment Training (GRPO)

In [ ]:
# Initialize reward engine
reward_engine = CompositeRewardEngine()

# Run GRPO alignment
print("Starting GRPO alignment training...")
aligned_model = run_grpo_training(
    model=trained_model,
    train_dataloader=None,  # Would be created from training data
    config=data_config,
    reward_engine=reward_engine
)

print("✓ GRPO alignment completed")


: 

## 6. Evaluation

In [ ]:
# Import evaluation
from evaluation.evaluator import MedicalTwinEvaluator

# Run evaluation
evaluator = MedicalTwinEvaluator()
results = evaluator.evaluate_model(aligned_model)

print("✓ Evaluation completed")
print(f"Results: {results}")


: 